In [ ]:
import os, sys, json, re
project_dir = f'/home/{os.environ["USER"]}/FireScope'
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)
from pathlib import Path
from typing import Optional, List, Dict

import numpy as np
from PIL import Image as PILImage
from tqdm import tqdm
from datasets import Dataset, Features, Value, Image, load_from_disk
import torch
from transformers import AutoModelForImageTextToText, Qwen2_5_VLProcessor
import random

# ---- Determinism settings ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
from prompts import main_prompt  # expects main_prompt.build_prompt(climate_dict)
DATA_DIR = '/work/wildfirerisk'
PREPROCESSED_DIR = os.path.join(DATA_DIR, "large_dataset", "preprocessed_hf")
TRAIN_DS_DIR = os.path.join(PREPROCESSED_DIR, "train")
# ===========================
# CONFIG: set everything here
# ===========================
DATASETS: List[Dict[str, str]] = [
    #load_from_disk(TRAIN_DS_DIR),
    {
        "img_dir": f"/work/wildfirerisk/europe_dataset/fire_images/{yr-1}",
        "raster_dir": f"/work/wildfirerisk/europe_dataset/fire_masks/{yr}"}
        for yr in range(2018, 2026)] + [
    {
        "img_dir": "/work/wildfirerisk/europe_dataset/no_fire_images",
        "raster_dir": "/work/wildfirerisk/europe_dataset/no_fire_masks",
    },
    {
        "img_dir": "/work/wildfirerisk/large_dataset/satellite_images_full/test",
        "raster_dir": "/work/wildfirerisk/large_dataset/normalised_risk_rasters/test",
    }
]

CLIMATE_JSON = "/work/wildfirerisk/climate_data.json"

VLM_CKPT = "/work/wildfirerisk/trainings/big/oracles/consistent_reward_cot/checkpoint-2600"
BASE_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
OUT_JSON = "/work/wildfirerisk/predictions/consistent_reward/2600_big.json"

BATCH_SIZE = 64
MAX_NEW_TOKENS = 1024
DEVICE_STR = "cuda"  # "cuda" or "cpu"
NUM_IMAGES_LIMIT = -1  # -1 for all

# ---------------------------
# Filename parsing for lat/lon
# ---------------------------
_COORD_RE = re.compile(r"lon(?P<lon>-?[0-9]+(?:\.[0-9]+)?)_lat(?P<lat>-?[0-9]+(?:\.[0-9]+)?)")
def parse_lat_lon_from_name(path: str) -> Optional[str]:
    m = _COORD_RE.search(Path(path).name)
    if not m:
        return None
    lon = float(m.group('lon'))
    lat = float(m.group('lat'))
    return f"{lat}_{lon}"

# ---------------------------
# Final answer parsing
# ---------------------------
FINAL_RE = re.compile(r"FINAL ANSWER:\s*([0-9])\s*$", re.IGNORECASE)
def parse_final_digit(text: str, default: int = 4) -> int:
    text = text.strip()
    m = FINAL_RE.search(text)
    if m:
        return int(m.group(1))
    # Fallback: last digit anywhere
    m2 = re.findall(r"([0-9])", text)
    return int(m2[-1]) if m2 else int(default)

# ---------------------------
# Prompt builder (matches GRPO flow)
# ---------------------------
def build_grpo_prompt(processor: Qwen2_5_VLProcessor, climate_dict: Dict) -> str:
    user_text = main_prompt.build_prompt(climate_dict)
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": user_text},
            ],
        },
    ]
    return processor.apply_chat_template(
        conversation, add_generation_prompt=True, tokenize=False
    )

# ---------------------------
# Batch helper
# ---------------------------
def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

# ---------------------------
# Ground truth helper
# ---------------------------
def compute_ground_truth_from_npy(npy_path: str) -> Optional[int]:
    if not Path(npy_path).is_file():
        return None
    arr = np.load(npy_path)
    # Expect a 1-channel "image" array; take mean over all elements
    mean_val = float(np.mean(arr))
    return min(int(mean_val * 10.0), 9)

# ---------------------------
# Main
# ---------------------------
def main():
    device = torch.device(DEVICE_STR if torch.cuda.is_available() else "cpu")
    outp = Path(OUT_JSON)
    outp.parent.mkdir(parents=True, exist_ok=True)
    # Load climate dict
    with open(CLIMATE_JSON, "r") as f:
        climate: Dict[str, Dict] = json.load(f)

    # Collect items from all dataset pairs
    items: List[Dict] = []
    total_skipped = 0
    total_missing_gt = 0

    for pair in DATASETS:
        if isinstance(pair, dict):
            img_dir = pair["img_dir"]
            raster_dir = pair["raster_dir"]

            img_paths = sorted([str(p) for p in Path(img_dir).glob("*.png")])
            if NUM_IMAGES_LIMIT > 0:
                img_paths = img_paths[:NUM_IMAGES_LIMIT]

            if not img_paths:
                print(f"[WARN] No PNGs found in {img_dir}")
                continue

            for ip in img_paths:
                key = parse_lat_lon_from_name(ip)
                if key is None or key not in climate:
                    total_skipped += 1
                    continue

                # Expect raster with same filename but .npy in raster_dir
                npy_name = Path(ip).name.replace(".png", ".npy")
                npy_path = str(Path(raster_dir) / npy_name)
                if not Path(npy_path).is_file():
                    print(npy_path)
                    total_missing_gt += 1
                    # We'll still run the model, but GT will be None
                    # (You can choose to skip instead; leaving it included is useful to compare later)
                items.append(
                    {
                        "path": ip,
                        "key": key,
                        "npy_path": npy_path,
                        "label": None
                    }
                )
        else:
            for item in pair:
                ip = item["image_path"]
                key = parse_lat_lon_from_name(ip)
                items.append(
                    {
                        "path": item["image_path"],
                        "key": key,
                        "label": item["solution"],
                        "npy_path": None
                    }
                )

    if not items:
        raise RuntimeError("No items remaining after filtering (climate keys and/or dataset contents).")

    if total_skipped:
        print(f"[INFO] Skipped {total_skipped} images due to missing/unparseable keys in climate JSON.")
    if total_missing_gt:
        print(f"[INFO] {total_missing_gt} images have no matching .npy raster; ground_truth will be null for those.")

    # Processor / Tokenizer
    processor = Qwen2_5_VLProcessor.from_pretrained(BASE_MODEL_ID, use_fast=True, padding_side='left')

    # Model (load FT checkpoint with base config)
    vlm = AutoModelForImageTextToText.from_pretrained(
        VLM_CKPT,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    ).to(device).eval()

    results: Dict[str, Dict[str, object]] = {}
    total = len(items)
    print(f"Running VLM on {total} items, batch_size={BATCH_SIZE}")
    cpt = 0
    with torch.inference_mode():
        for (i, batch) in enumerate(tqdm(list(chunks(items, BATCH_SIZE)), total=(total + BATCH_SIZE - 1)//BATCH_SIZE)):
            pil_images = []
            prompts = []
            keys = []
            npy_paths = []
            img_paths = []
            raster_paths = []
            labels = []
            for it in batch:
                img = PILImage.open(it["path"]).convert("RGB")
                img_paths.append(it['path'])
                pil_images.append(img)
                prompts.append(build_grpo_prompt(processor, climate[it["key"]]))
                keys.append(it["key"])
                npy_paths.append(it["npy_path"])
                labels.append(it['label'])

            inputs = processor(images=pil_images, text=prompts, padding=True, return_tensors="pt")
            inputs = {k: (v.to(device) if hasattr(v, "to") else v) for k, v in inputs.items()}

            gen_ids = vlm.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
            texts = processor.tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

            # Parse digits and compute GTs
            digits = [parse_final_digit(t, default=-1) for t in texts]
            gts = []
            for j, npy_path in enumerate(npy_paths):
                if npy_path:
                    gts.append(compute_ground_truth_from_npy(npy_path))
                else:
                    gts.append(labels[j])

            for k, t, d, gt, ip, rp in zip(keys, texts, digits, gts, img_paths, npy_paths):
                results[k] = {
                    "model_response": t,
                    "answer": int(d),
                    "ground_truth": (None if gt is None else int(gt)),
                    "img_path": ip,
                    "raster_path": rp
                }

            # free a bit
            del inputs, gen_ids, pil_images, prompts
            if i > cpt + 5:
                with open(outp, "w") as f:
                    pass
                    json.dump(results, f, indent=2)
                cpt = i

    # Save
    with open(outp, "w") as f:
        pass
        json.dump(results, f, indent=2)
    print(f"Saved {len(results)} entries to {outp}")

if __name__ == "__main__":
    main()


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Running VLM on 8924 items, batch_size=64


100%|██████████| 140/140 [3:30:31<00:00, 90.23s/it] 


Saved 8924 entries to /work/wildfirerisk/predictions/consistent_reward/2600_big.json


In [ ]:
import os, sys, json, re
project_dir = f'/home/{os.environ["USER"]}/FireScope'
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)
from pathlib import Path
from typing import Optional, List, Dict

import numpy as np
from PIL import Image as PILImage
from tqdm import tqdm
from datasets import Dataset, Features, Value, Image, load_from_disk
import torch
from transformers import AutoModelForImageTextToText, Qwen2_5_VLProcessor
import random

# ---- Determinism settings ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
from prompts import main_prompt  # expects main_prompt.build_prompt(climate_dict)
DATA_DIR = '/work/wildfirerisk'
PREPROCESSED_DIR = os.path.join(DATA_DIR, "large_dataset", "preprocessed_hf")
TRAIN_DS_DIR = os.path.join(PREPROCESSED_DIR, "train")
# ===========================
# CONFIG: set everything here
# ===========================
DATASETS: List[Dict[str, str]] = [
    #load_from_disk(TRAIN_DS_DIR),
    {
        "img_dir": f"/work/wildfirerisk/europe_dataset/fire_images/{yr-1}",
        "raster_dir": f"/work/wildfirerisk/europe_dataset/fire_masks/{yr}"}
        for yr in range(2018, 2026)] + [
    {
        "img_dir": "/work/wildfirerisk/europe_dataset/no_fire_images",
        "raster_dir": "/work/wildfirerisk/europe_dataset/no_fire_masks",
    },
    {
        "img_dir": "/work/wildfirerisk/large_dataset/satellite_images_full/test",
        "raster_dir": "/work/wildfirerisk/large_dataset/normalised_risk_rasters/test",
    }
]

CLIMATE_JSON = "/work/wildfirerisk/climate_data.json"

VLM_CKPT = "/work/wildfirerisk/trainings/big/oracles/qwen_vlm_grpo/checkpoint-1400"
BASE_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
OUT_JSON = "/work/wildfirerisk/vlm_cot_big_checkpoint_1400.json"

BATCH_SIZE = 64
MAX_NEW_TOKENS = 1024
DEVICE_STR = "cuda"  # "cuda" or "cpu"
NUM_IMAGES_LIMIT = -1  # -1 for all

# ---------------------------
# Filename parsing for lat/lon
# ---------------------------
_COORD_RE = re.compile(r"lon(?P<lon>-?[0-9]+(?:\.[0-9]+)?)_lat(?P<lat>-?[0-9]+(?:\.[0-9]+)?)")
def parse_lat_lon_from_name(path: str) -> Optional[str]:
    m = _COORD_RE.search(Path(path).name)
    if not m:
        return None
    lon = float(m.group('lon'))
    lat = float(m.group('lat'))
    return f"{lat}_{lon}"

# ---------------------------
# Final answer parsing
# ---------------------------
FINAL_RE = re.compile(r"FINAL ANSWER:\s*([0-9])\s*$", re.IGNORECASE)
def parse_final_digit(text: str, default: int = 4) -> int:
    text = text.strip()
    m = FINAL_RE.search(text)
    if m:
        return int(m.group(1))
    # Fallback: last digit anywhere
    m2 = re.findall(r"([0-9])", text)
    return int(m2[-1]) if m2 else int(default)

# ---------------------------
# Prompt builder (matches GRPO flow)
# ---------------------------
def build_grpo_prompt(processor: Qwen2_5_VLProcessor, climate_dict: Dict) -> str:
    user_text = main_prompt.build_prompt(climate_dict)
    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": user_text},
            ],
        },
    ]
    return processor.apply_chat_template(
        conversation, add_generation_prompt=True, tokenize=False
    )

# ---------------------------
# Batch helper
# ---------------------------
def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

# ---------------------------
# Ground truth helper
# ---------------------------
def compute_ground_truth_from_npy(npy_path: str) -> Optional[int]:
    if not Path(npy_path).is_file():
        return None
    arr = np.load(npy_path)
    # Expect a 1-channel "image" array; take mean over all elements
    mean_val = float(np.mean(arr))
    return min(int(mean_val * 10.0), 9)

# ---------------------------
# Main
# ---------------------------
def main():
    device = torch.device(DEVICE_STR if torch.cuda.is_available() else "cpu")
    outp = Path(OUT_JSON)
    outp.parent.mkdir(parents=True, exist_ok=True)
    # Load climate dict
    with open(CLIMATE_JSON, "r") as f:
        climate: Dict[str, Dict] = json.load(f)

    # Collect items from all dataset pairs
    items: List[Dict] = []
    total_skipped = 0
    total_missing_gt = 0

    for pair in DATASETS:
        if isinstance(pair, dict):
            img_dir = pair["img_dir"]
            raster_dir = pair["raster_dir"]

            img_paths = sorted([str(p) for p in Path(img_dir).glob("*.png")])
            if NUM_IMAGES_LIMIT > 0:
                img_paths = img_paths[:NUM_IMAGES_LIMIT]

            if not img_paths:
                print(f"[WARN] No PNGs found in {img_dir}")
                continue

            for ip in img_paths:
                key = parse_lat_lon_from_name(ip)
                if key is None or key not in climate:
                    total_skipped += 1
                    continue

                # Expect raster with same filename but .npy in raster_dir
                npy_name = Path(ip).name.replace(".png", ".npy")
                npy_path = str(Path(raster_dir) / npy_name)
                if not Path(npy_path).is_file():
                    print(npy_path)
                    total_missing_gt += 1
                    # We'll still run the model, but GT will be None
                    # (You can choose to skip instead; leaving it included is useful to compare later)
                items.append(
                    {
                        "path": ip,
                        "key": key,
                        "npy_path": npy_path,
                        "label": None
                    }
                )
        else:
            for item in pair:
                ip = item["image_path"]
                key = parse_lat_lon_from_name(ip)
                items.append(
                    {
                        "path": item["image_path"],
                        "key": key,
                        "label": item["solution"],
                        "npy_path": None
                    }
                )

    if not items:
        raise RuntimeError("No items remaining after filtering (climate keys and/or dataset contents).")

    if total_skipped:
        print(f"[INFO] Skipped {total_skipped} images due to missing/unparseable keys in climate JSON.")
    if total_missing_gt:
        print(f"[INFO] {total_missing_gt} images have no matching .npy raster; ground_truth will be null for those.")

    # Processor / Tokenizer
    processor = Qwen2_5_VLProcessor.from_pretrained(BASE_MODEL_ID, use_fast=True, padding_side='left')

    # Model (load FT checkpoint with base config)
    vlm = AutoModelForImageTextToText.from_pretrained(
        VLM_CKPT,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    ).to(device).eval()

    results: Dict[str, Dict[str, object]] = {}
    total = len(items)
    print(f"Running VLM on {total} items, batch_size={BATCH_SIZE}")
    cpt = 0
    with torch.inference_mode():
        for (i, batch) in enumerate(tqdm(list(chunks(items, BATCH_SIZE)), total=(total + BATCH_SIZE - 1)//BATCH_SIZE)):
            pil_images = []
            prompts = []
            keys = []
            npy_paths = []
            img_paths = []
            raster_paths = []
            labels = []
            for it in batch:
                img = PILImage.open(it["path"]).convert("RGB")
                img_paths.append(it['path'])
                pil_images.append(img)
                prompts.append(build_grpo_prompt(processor, climate[it["key"]]))
                keys.append(it["key"])
                npy_paths.append(it["npy_path"])
                labels.append(it['label'])

            inputs = processor(images=pil_images, text=prompts, padding=True, return_tensors="pt")
            inputs = {k: (v.to(device) if hasattr(v, "to") else v) for k, v in inputs.items()}

            gen_ids = vlm.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
            texts = processor.tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

            # Parse digits and compute GTs
            digits = [parse_final_digit(t, default=-1) for t in texts]
            gts = []
            for j, npy_path in enumerate(npy_paths):
                if npy_path:
                    gts.append(compute_ground_truth_from_npy(npy_path))
                else:
                    gts.append(labels[j])

            for k, t, d, gt, ip, rp in zip(keys, texts, digits, gts, img_paths, npy_paths):
                results[k] = {
                    "model_response": t,
                    "answer": int(d),
                    "ground_truth": (None if gt is None else int(gt)),
                    "img_path": ip,
                    "raster_path": rp
                }

            # free a bit
            del inputs, gen_ids, pil_images, prompts
            if i > cpt + 5:
                with open(outp, "w") as f:
                    pass
                    json.dump(results, f, indent=2)
                cpt = i

    # Save
    with open(outp, "w") as f:
        pass
        json.dump(results, f, indent=2)
    print(f"Saved {len(results)} entries to {outp}")

if __name__ == "__main__":
    main()


In [7]:
import json
with open("/work/wildfirerisk/vlm_cot_big_checkpoint_4800.json", 'r') as f:
    x = json.load(f)
for k, v in x.items():
    if not v['raster_path']:
        continue
    if 'risk_rasters' in v['raster_path'] and 'val' in v['raster_path']:
        print(v['raster_path'])
        break

/work/wildfirerisk/small_dataset/normalised_risk_rasters/val/tile_AL_val_lon-88.1401_lat30.6273.npy
